# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [37]:
%pip install matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [39]:
df = pd.read_csv("data/AviationData.csv",encoding="cp1252", low_memory=False)

In [40]:

# 1. Inspect the number of rows and columns
print("Dataset shape:", df.shape)

# 2. Inspect column names
print("\nColumn names:")
print(df.columns.tolist())

# 3. Inspect data types and non-null counts
print("\nDataFrame information:")
df.info()

# 4. Inspect missing values
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2),
    "Data_Type": df.dtypes.astype(str)
})

display(
    missing_summary.sort_values(
        by="Missing_Percentage",
        ascending=False
    )
)

# 5. Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# 6. Summary statistics for numerical and categorical columns
print("\nSummary statistics:")
display(df.describe(include="all").T)

Dataset shape: (88889, 31)

Column names:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date']

DataFrame information:
<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date             

,Missing_Count,Missing_Percentage,Data_Type
Schedule,76307,85.85,str
Air.carrier,72241,81.27,str
FAR.Description,56866,63.97,str
Aircraft.Category,56602,63.68,str
Longitude,54516,61.33,str
Latitude,54507,61.32,str
Airport.Code,38757,43.60,str
Airport.Name,36185,40.71,str
Broad.phase.of.flight,27165,30.56,str
Publication.Date,13771,15.49,str



Duplicate rows: 0

Summary statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Event.Id,88889,87951,20001214X45071,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Investigation.Type,88889,2,Accident,85015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident.Number,88889,88863,ERA22LA103,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event.Date,88889,14782,1982-05-16,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,88837,27758,"ANCHORAGE, AK",434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,88663,219,United States,82248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,34382,25589,332739N,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Longitude,34373,27154,0112457W,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Code,50132,10374,NONE,1488,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Name,52704,24870,Private,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [41]:

# Make a copy so the original dataset remains unchanged
aviation_df = df.copy()

# Convert accident date to datetime
aviation_df["Event.Date"] = pd.to_datetime(
    aviation_df["Event.Date"],
    errors="coerce"
)

# Standardize text fields
for col in ["Make", "Model", "Amateur.Built", "Aircraft.Category"]:
    if col in aviation_df.columns:
        aviation_df[col] = aviation_df[col].replace(
            r"^\s*$", np.nan, regex=True
        )

# Filter accident dates from 1983 through 2023
filtered_df = aviation_df[
    aviation_df["Event.Date"].between(
        "1983-01-01", "2023-12-31"
    )
].copy()

# Keep professionally built aircraft
filtered_df = filtered_df[
    filtered_df["Amateur.Built"].astype("string").str.strip().str.lower()
    == "no"
].copy()

# Keep records with known aircraft make and model
filtered_df = filtered_df.dropna(subset=["Make", "Model"])

# Keep confirmed airplanes, if the category column exists
if "Aircraft.Category" in filtered_df.columns:
    filtered_df = filtered_df[
        filtered_df["Aircraft.Category"].astype("string").str.strip().str.lower()
        == "airplane"
    ].copy()

# Check the filtered dataset
print("Original records:", len(df))
print("Filtered records:", len(filtered_df))
print("Records removed:", len(df) - len(filtered_df))

display(filtered_df.head())


Original records: 88889
Filtered records: 21428
Records removed: 67461


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
4149,20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,Unknown,NaN,NaN,NaN,NaN,588.0,VMC,Standing,Probable Cause,04-12-2014
4150,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,NaN,"Singapore Airlines, Ltd.",NaN,NaN,NaN,588.0,VMC,Taxi,Probable Cause,04-12-2014
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,1.0,NaN,NaN,IMC,Cruise,Probable Cause,02-05-2011
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,Skydiving,NaN,1.0,NaN,NaN,4.0,VMC,Standing,Probable Cause,17-10-2016
5957,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,...,Unknown,NaN,NaN,NaN,NaN,289.0,VMC,Cruise,Probable Cause,01-02-2016


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [42]:
# Inspect columns relevant to injuries, passengers, and aircraft damage
relevant_columns = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
    "Aircraft.damage",
    "Injury.Severity",
    "Number.of.Engines",
    "Make",
    "Model"
]

# Display only columns that exist in the dataset
available_columns = [
    col for col in relevant_columns
    if col in filtered_df.columns
]

display(filtered_df[available_columns].head())
display(filtered_df[available_columns].isna().sum())

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Aircraft.damage,Injury.Severity,Number.of.Engines,Make,Model
4149,NaN,NaN,NaN,588.0,Minor,Incident,3.0,Lockheed,L-1011
4150,NaN,NaN,NaN,588.0,Minor,Incident,4.0,Boeing,747
4171,1.0,1.0,NaN,NaN,Destroyed,Fatal(1),1.0,Piper,PA-28-140
4285,1.0,NaN,NaN,4.0,NaN,Fatal(1),2.0,De Havilland,DHC-6
5957,NaN,NaN,NaN,289.0,Minor,Incident,3.0,Douglas,DC-10-10


Total.Fatal.Injuries      2744
Total.Serious.Injuries    2820
Total.Minor.Injuries      2537
Total.Uninjured            705
Aircraft.damage           1224
Injury.Severity            811
Number.of.Engines         2546
Make                         0
Model                        0
dtype: int64

In [43]:
# Work on a copy so the original filtered dataset remains unchanged
safety_df = filtered_df.copy()

# Identify injury-count columns that are available
injury_columns = [
    col for col in [
        "Total.Fatal.Injuries",
        "Total.Serious.Injuries",
        "Total.Minor.Injuries",
        "Total.Uninjured"
    ]
    if col in safety_df.columns
]

# Convert injury counts to numeric values.
# Invalid text becomes NaN rather than being treated as zero.
for col in injury_columns:
    safety_df[col] = pd.to_numeric(safety_df[col], errors="coerce")

    # Negative injury counts are invalid, so mark them as missing.
    safety_df.loc[safety_df[col] < 0, col] = np.nan

# Standardize aircraft-damage text
if "Aircraft.damage" in safety_df.columns:
    safety_df["Aircraft.damage"] = (
        safety_df["Aircraft.damage"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    # Treat blank entries as missing
    safety_df["Aircraft.damage"] = safety_df["Aircraft.damage"].replace(
        "", pd.NA
    )

    # Impute missing damage labels as Unknown.
    # This preserves records without pretending the aircraft was undamaged.
    safety_df["Aircraft.damage"] = (
        safety_df["Aircraft.damage"].fillna("Unknown")
    )

# Report missing injury counts after cleaning
display(safety_df[injury_columns].isna().sum())

Total.Fatal.Injuries      2744
Total.Serious.Injuries    2820
Total.Minor.Injuries      2537
Total.Uninjured            705
dtype: int64

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [44]:
# Create a copy of the filtered dataset
safety_df = filtered_df.copy()

# Inspect existing aircraft damage categories
print(safety_df["Aircraft.damage"].value_counts(dropna=False))

# Remove leading/trailing spaces and standardize capitalization
safety_df["Aircraft.damage"] = (
    safety_df["Aircraft.damage"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Convert blank strings to missing values
safety_df["Aircraft.damage"] = (
    safety_df["Aircraft.damage"].replace("", pd.NA)
)

# Check the cleaned categories
print(safety_df["Aircraft.damage"].value_counts(dropna=False))

Aircraft.damage
Substantial    16982
Destroyed       2311
NaN             1224
Minor            814
Unknown           97
Name: count, dtype: int64
Aircraft.damage
Substantial    16982
Destroyed       2311
<NA>            1224
Minor            814
Unknown           97
Name: count, dtype: Int64


In [ ]:
safety_df["Aircraft.Destroyed"] = (
    safety_df["Aircraft.damage"]
    .map({
        "Destroyed": 1,
        "Substantial": 0,
        "Minor": 0,
        "None": 0
    })
)

# Check the results
display(
    safety_df[
        ["Aircraft.damage", "Aircraft.Destroyed"]
    ].head(10)
)

# Count destroyed, not destroyed, and unknown cases
print(safety_df["Aircraft.Destroyed"].value_counts(dropna=False))


,Aircraft.damage,Aircraft.Destroyed
4149,Minor,0.0
4150,Minor,0.0
4171,Destroyed,1.0
4285,<NA>,NaN
5957,Minor,0.0
5960,Destroyed,1.0
6669,<NA>,NaN
6760,Minor,0.0
6806,Substantial,0.0
7084,Substantial,0.0


Aircraft.Destroyed
0.0    17796
1.0     2311
NaN     1321
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [46]:
# Keep the original filtered dataset unchanged
cleaned_df = filtered_df.copy()

# 1. Standardize Make and Model
for col in ["Make", "Model"]:
    cleaned_df[col] = (
        cleaned_df[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    # Treat blank strings as missing values
    cleaned_df[col] = cleaned_df[col].replace("", pd.NA)

# 2. Remove records with missing Make or Model
cleaned_df = cleaned_df.dropna(
    subset=["Make", "Model"]
).copy()

# 3. Standardize aircraft damage categories
if "Aircraft.damage" in cleaned_df.columns:
    cleaned_df["Aircraft.damage"] = (
        cleaned_df["Aircraft.damage"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    cleaned_df["Aircraft.damage"] = (
        cleaned_df["Aircraft.damage"].replace("", pd.NA)
    )

# 4. Validate injury-count columns
injury_columns = [
    col for col in [
        "Total.Fatal.Injuries",
        "Total.Serious.Injuries",
        "Total.Minor.Injuries",
        "Total.Uninjured"
    ]
    if col in cleaned_df.columns
]

for col in injury_columns:
    cleaned_df[col] = pd.to_numeric(
        cleaned_df[col], errors="coerce"
    )

    # Negative injury counts are invalid
    cleaned_df.loc[cleaned_df[col] < 0, col] = np.nan

# 5. Remove exact duplicate rows
before_duplicates = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates().copy()
duplicates_removed = before_duplicates - len(cleaned_df)

# 6. Create the aircraft destruction indicator
# 1 = destroyed; 0 = another known damage category;
# missing = damage unknown or unrecognized.
if "Aircraft.damage" in cleaned_df.columns:
    damage_map = {
        "Destroyed": 1,
        "Substantial": 0,
        "Minor": 0,
        "None": 0
    }

    cleaned_df["Aircraft.Destroyed"] = (
        cleaned_df["Aircraft.damage"].map(damage_map)
    )

# 7. Retain makes with at least 40 records
make_counts = cleaned_df["Make"].value_counts()

eligible_makes = make_counts[
    make_counts >= 40
].index

analysis_df = cleaned_df[
    cleaned_df["Make"].isin(eligible_makes)
].copy()

# 8. Report cleaning results
print("Records before cleaning:", len(filtered_df))
print("Records after cleaning, before make threshold:", len(cleaned_df))
print("Exact duplicate rows removed:", duplicates_removed)
print("Number of eligible makes:", len(eligible_makes))
print("Records after applying the 50-record threshold:", len(analysis_df))

# Display the most frequently represented makes
display(analysis_df["Make"].value_counts().head(20))

# Inspect the final dataset
display(analysis_df.head())

Records before cleaning: 21428
Records after cleaning, before make threshold: 21428
Exact duplicate rows removed: 0
Number of eligible makes: 45
Records after applying the 50-record threshold: 18278


Make
CESSNA                7142
PIPER                 3988
BEECH                 1431
BOEING                1261
MOONEY                 363
AIRBUS                 243
CIRRUS DESIGN CORP     220
BELLANCA               219
AIR TRACTOR INC        219
MAULE                  215
AIR TRACTOR            206
AERONCA                200
CHAMPION               158
EMBRAER                152
GRUMMAN                147
LUSCOMBE               141
CIRRUS                 137
STINSON                129
MCDONNELL DOUGLAS      108
NORTH AMERICAN         106
Name: count, dtype: Int64

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Aircraft.Destroyed
4150,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,"Singapore Airlines, Ltd.",NaN,NaN,NaN,588.0,VMC,Taxi,Probable Cause,04-12-2014,0.0
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,NaN,1.0,1.0,NaN,NaN,IMC,Cruise,Probable Cause,02-05-2011,1.0
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,NaN,1.0,NaN,NaN,4.0,VMC,Standing,Probable Cause,17-10-2016,NaN
5957,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,289.0,VMC,Cruise,Probable Cause,01-02-2016,0.0
6760,20001214X45013,Incident,CHI84IA041,1983-11-08,"CHICAGO, IL",United States,NaN,NaN,ORD,O'HARE,...,NaN,NaN,NaN,NaN,100.0,VMC,Taxi,Probable Cause,11-06-2018,0.0


In [47]:
# Check missing values in important columns
important_columns = [
    col for col in [
        "Make",
        "Model",
        "Aircraft.damage",
        "Aircraft.Destroyed",
        "Total.Fatal.Injuries",
        "Total.Serious.Injuries",
        "Total.Minor.Injuries",
        "Total.Uninjured"
    ]
    if col in analysis_df.columns
]

display(analysis_df[important_columns].isna().sum())

# Confirm that every retained make has at least 50 records
final_make_counts = analysis_df["Make"].value_counts()

print("Smallest make count:", final_make_counts.min())
print("All retained makes meet the threshold:",
      (final_make_counts >= 50).all())

Make                         0
Model                        0
Aircraft.damage           1094
Aircraft.Destroyed        1176
Total.Fatal.Injuries      2433
Total.Serious.Injuries    2506
Total.Minor.Injuries      2270
Total.Uninjured            599
dtype: int64

Smallest make count: 41
All retained makes meet the threshold: False


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [48]:
safety_df = safety_df.dropna().copy()

In [49]:
# Count records for each aircraft make
make_counts = safety_df["Make"].value_counts()
display(make_counts.head(20))

# Count records for each model
model_counts = safety_df["Model"].value_counts()
display(model_counts.head(20))

# Check whether each model appears under multiple makes
model_make_counts = (
    safety_df.groupby("Model")["Make"]
    .nunique()
    .sort_values(ascending=False)
)

display(model_make_counts.head(20))

# Models associated with more than one make
print("Models associated with multiple makes:")
display(model_make_counts[model_make_counts > 1])

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)

Series([], Name: Make, dtype: int64)

Models associated with multiple makes:


Series([], Name: Make, dtype: int64)

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [50]:
# Columns to inspect
columns_to_check = [
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

# Inspect only columns that exist
for col in columns_to_check:
    if col in analysis_df.columns:
        print(f"\n--- {col} ---")
        print("Data type:", analysis_df[col].dtype)
        print("Missing values:", analysis_df[col].isna().sum())
        display(analysis_df[col].value_counts(dropna=False).head(30))
    else:
        print(f"{col} is not present in the DataFrame.")


--- Engine.Type ---
Data type: str
Missing values: 3291


Engine.Type
Reciprocating      13028
NaN                 3291
Turbo Prop           980
Turbo Fan            752
Unknown              116
Turbo Jet             88
Geared Turbofan       12
Turbo Shaft           10
UNK                    1
Name: count, dtype: int64


--- Weather.Condition ---
Data type: str
Missing values: 2468


Weather.Condition
VMC    14605
NaN     2468
IMC      926
Unk      194
UNK       85
Name: count, dtype: int64


--- Number.of.Engines ---
Data type: float64
Missing values: 2150


Number.of.Engines
1.0    13458
2.0     2560
NaN     2150
4.0       76
3.0       28
0.0        6
Name: count, dtype: int64


--- Purpose.of.flight ---
Data type: str
Missing values: 3147


Purpose.of.flight
Personal                     9987
NaN                          3147
Instructional                2455
Aerial Application            774
Business                      417
Unknown                       321
Positioning                   279
Skydiving                     157
Aerial Observation            150
Other Work Use                129
Banner Tow                     86
Flight Test                    78
Ferry                          73
Executive/corporate            67
Public Aircraft - Federal      30
Glider Tow                     29
Public Aircraft                27
Public Aircraft - State        22
Air Race show                  16
Firefighting                   13
Public Aircraft - Local        12
PUBS                            3
Air Race/show                   2
Air Drop                        2
ASHO                            2
Name: count, dtype: int64


--- Broad.phase.of.flight ---
Data type: str
Missing values: 15785


Broad.phase.of.flight
NaN            15785
Landing         1127
Takeoff          430
Cruise           243
Approach         213
Maneuvering      131
Taxi             101
Go-around         82
Descent           63
Climb             52
Standing          38
Unknown           11
Other              2
Name: count, dtype: int64

In [51]:
# Create a copy to preserve the original analysis dataset
cleaned_analysis_df = analysis_df.copy()

# Categorical columns to standardize
categorical_columns = [
    "Engine.Type",
    "Weather.Condition",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

# Common labels that represent unknown or unreported information
unknown_labels = [
    "unknown",
    "unk",
    "n/a",
    "na",
    "not reported",
    "not known",
    "undetermined"
]

# Clean categorical columns
for col in categorical_columns:
    if col in cleaned_analysis_df.columns:

        # Convert to pandas string type and remove extra spaces
        cleaned_analysis_df[col] = (
            cleaned_analysis_df[col]
            .astype("string")
            .str.strip()
        )

        # Treat blank strings as missing
        cleaned_analysis_df[col] = (
            cleaned_analysis_df[col].replace("", pd.NA)
        )

        # Standardize capitalization
        cleaned_analysis_df[col] = (
            cleaned_analysis_df[col].str.title()
        )

        # Consolidate common unknown labels
        cleaned_analysis_df[col] = cleaned_analysis_df[col].replace(
            {label.title(): "Unknown" for label in unknown_labels}
        )

# Clean Number.of.Engines
if "Number.of.Engines" in cleaned_analysis_df.columns:

    # Convert to numeric; invalid text becomes missing
    cleaned_analysis_df["Number.of.Engines"] = pd.to_numeric(
        cleaned_analysis_df["Number.of.Engines"],
        errors="coerce"
    )

    # Negative engine counts are invalid
    cleaned_analysis_df.loc[
        cleaned_analysis_df["Number.of.Engines"] < 0,
        "Number.of.Engines"
    ] = np.nan

    # Use nullable integer format while preserving missing values
    cleaned_analysis_df["Number.of.Engines"] = (
        cleaned_analysis_df["Number.of.Engines"].astype("Int64")
    )

print("Cleaning completed.")

Cleaning completed.


In [52]:
# Inspect cleaned categories and counts
for col in columns_to_check:
    if col in cleaned_analysis_df.columns:
        print(f"\n--- Cleaned: {col} ---")
        display(
            cleaned_analysis_df[col]
            .value_counts(dropna=False)
            .head(30)
        )

# Summarize remaining missing values
available_columns = [
    col for col in columns_to_check
    if col in cleaned_analysis_df.columns
]

display(
    cleaned_analysis_df[available_columns]
    .isna()
    .sum()
    .rename("Missing Values")
    .to_frame()
)

# Confirm data types
print(cleaned_analysis_df[available_columns].dtypes)


--- Cleaned: Engine.Type ---


Engine.Type
Reciprocating      13028
<NA>                3291
Turbo Prop           980
Turbo Fan            752
Unknown              117
Turbo Jet             88
Geared Turbofan       12
Turbo Shaft           10
Name: count, dtype: Int64


--- Cleaned: Weather.Condition ---


Weather.Condition
Vmc        14605
<NA>        2468
Imc          926
Unknown      279
Name: count, dtype: Int64


--- Cleaned: Number.of.Engines ---


...

Purpose.of.flight
Personal                     9987
<NA>                         3147
Instructional                2455
Aerial Application            774
Business                      417
Unknown                       321
Positioning                   279
Skydiving                     157
Aerial Observation            150
Other Work Use                129
Banner Tow                     86
Flight Test                    78
Ferry                          73
Executive/Corporate            67
Public Aircraft - Federal      30
Glider Tow                     29
Public Aircraft                27
Public Aircraft - State        22
Air Race Show                  16
Firefighting                   13
Public Aircraft - Local        12
Pubs                            3
Air Race/Show                   2
Air Drop                        2
Asho                            2
Name: count, dtype: Int64


--- Cleaned: Broad.phase.of.flight ---


Broad.phase.of.flight
<NA>           15785
Landing         1127
Takeoff          430
Cruise           243
Approach         213
Maneuvering      131
Taxi             101
Go-Around         82
Descent           63
Climb             52
Standing          38
Unknown           11
Other              2
Name: count, dtype: Int64

,Missing Values
Engine.Type,3291
Weather.Condition,2468
Number.of.Engines,2150
Purpose.of.flight,3147
Broad.phase.of.flight,15785


Engine.Type              string
Weather.Condition        string
Number.of.Engines         Int64
Purpose.of.flight        string
Broad.phase.of.flight    string
dtype: object


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [53]:
# Count missing values in each column
missing_count = cleaned_analysis_df.isna().sum()

# Calculate percentage of missing values
missing_percent = (
    cleaned_analysis_df.isna().mean() * 100
).round(2)

# Create a summary table
missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_percent
})

# Sort columns by percentage of missing values
missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

display(missing_summary)

,Missing_Count,Missing_Percentage
Schedule,16059,87.86
Broad.phase.of.flight,15785,86.36
Air.carrier,9599,52.52
Airport.Code,6372,34.86
Airport.Name,6268,34.29
Report.Status,3869,21.17
Engine.Type,3291,18.01
Purpose.of.flight,3147,17.22
Total.Serious.Injuries,2506,13.71
Weather.Condition,2468,13.50


In [54]:
# Set the missing-value threshold
threshold = 50

# Identify columns exceeding the threshold
columns_to_drop = missing_summary[
    missing_summary["Missing_Percentage"] > threshold
].index.tolist()

print("Columns to drop:")
print(columns_to_drop)

# Drop those columns
cleaned_analysis_df = cleaned_analysis_df.drop(
    columns=columns_to_drop
)

# Check the new DataFrame shape
print("Updated DataFrame shape:", cleaned_analysis_df.shape)

Columns to drop:
['Schedule', 'Broad.phase.of.flight', 'Air.carrier']
Updated DataFrame shape: (18278, 29)


In [55]:
# Check missing-value percentages after dropping columns
remaining_missing = (
    cleaned_analysis_df.isna().mean() * 100
).round(2).sort_values(ascending=False)

display(remaining_missing.to_frame(
    name="Missing_Percentage"
))

,Missing_Percentage
Airport.Code,34.86
Airport.Name,34.29
Report.Status,21.17
Engine.Type,18.01
Purpose.of.flight,17.22
Total.Serious.Injuries,13.71
Weather.Condition,13.50
Total.Fatal.Injuries,13.31
Total.Minor.Injuries,12.42
Number.of.Engines,11.76


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [56]:
# Save the cleaned DataFrame to a CSV file
cleaned_analysis_df.to_csv(
    "cleaned_aviation_data.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [57]:
# Load the previously cleaned dataset
aviation_df = pd.read_csv("cleaned_aviation_data.csv")

# Inspect the loaded dataset
display(aviation_df.head())

# Check the dimensions
print("Rows and columns:", aviation_df.shape)

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,FAR.Description,Purpose.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Report.Status,Publication.Date,Aircraft.Destroyed
0,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,Part 129: Foreign,NaN,NaN,NaN,NaN,588.0,Vmc,Probable Cause,04-12-2014,0.0
1,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,Part 91: General Aviation,Personal,1.0,1.0,NaN,NaN,Imc,Probable Cause,02-05-2011,1.0
2,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,Part 91: General Aviation,Skydiving,1.0,NaN,NaN,4.0,Vmc,Probable Cause,17-10-2016,NaN
3,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,...,Part 121: Air Carrier,Unknown,NaN,NaN,NaN,289.0,Vmc,Probable Cause,01-02-2016,0.0
4,20001214X45013,Incident,CHI84IA041,1983-11-08,"CHICAGO, IL",United States,NaN,NaN,ORD,O'HARE,...,Part 121: Air Carrier,Unknown,NaN,NaN,NaN,100.0,Vmc,Probable Cause,11-06-2018,0.0


Rows and columns: (18278, 29)


In [58]:
print("Original cleaned shape:", cleaned_analysis_df.shape)
print("Reloaded shape:", aviation_df.shape)

assert cleaned_analysis_df.shape == aviation_df.shape

print("Verification successful!")

Original cleaned shape: (18278, 29)
Reloaded shape: (18278, 29)
Verification successful!
